# 撤稿论文引用分析 — 模块(一): 找到 Focal Paper (i)

**日期:** 2026-05-12
**目标:** 从全量引用关系中筛出引用了撤稿文献的论文
**策略:** DuckDB 直接 join 52G CSV，不加载进 pandas
**预计耗时:** 5-10 分钟

In [12]:
import duckdb
import pandas as pd
import time
import os

con = duckdb.connect()
con.execute("SET memory_limit = '100GB'")
print("DuckDB 已连接, 内存上限 100G")

DuckDB 已连接, 内存上限 100G


In [13]:
# ====== 断点恢复: 如果已跑过 Step 1-2, 直接从磁盘加载 ======
import os
processed_dir = '/Data4/yutao_wen/processed'

# 检查 Step 1 输出
step1_file = os.path.join(processed_dir, 'df_containretra.csv')
step2_file = os.path.join(processed_dir, 'df_containretra2.csv')

if os.path.exists(step1_file):
    print(f'发现 Step 1 输出: {step1_file}')
    size_mb = os.path.getsize(step1_file) / 1024 / 1024
    print(f'  大小: {size_mb:.1f} MB, 跳过扫描, 直接加载')
    con.execute(f"DROP TABLE IF EXISTS df_containretra")
    con.execute(f"CREATE OR REPLACE TABLE df_containretra AS SELECT * FROM read_csv_auto('{step1_file}')")
    count = con.execute('SELECT COUNT(*) FROM df_containretra').fetchone()[0]
    unique_i = con.execute('SELECT COUNT(DISTINCT id) FROM df_containretra').fetchone()[0]
    print(f'  已加载: {count:,} 行, {unique_i:,} unique focal')

if os.path.exists(step2_file):
    print(f'\\n发现 Step 2 输出: {step2_file}')
    size_mb = os.path.getsize(step2_file) / 1024 / 1024
    print(f'  大小: {size_mb:.1f} MB, 直接加载')
    con.execute(f"DROP TABLE IF EXISTS df_containretra2_final")
    con.execute(f"CREATE OR REPLACE TABLE df_containretra2_final AS SELECT * FROM read_csv_auto('{step2_file}')")
    n = con.execute('SELECT COUNT(*) FROM df_containretra2_final').fetchone()[0]
    print(f'  已加载: {n:,} 行')
else:
    print('未找到已有输出, 将从头运行 Step 2')

print('\\n准备就绪! 可以从当前步骤继续运行')


发现 Step 1 输出: /Data4/yutao_wen/processed/df_containretra.csv
  大小: 21.9 MB, 跳过扫描, 直接加载
  已加载: 765,839 行, 647,439 unique focal
未找到已有输出, 将从头运行 Step 2
\n准备就绪! 可以从当前步骤继续运行


## Step 1/4: 加载撤稿论文名单

matched_output2.csv (50MB, 约4万篇撤稿论文)

In [14]:
retra_path = '/Data4/yutao_wen/matched_output2.csv'

con.execute(f"CREATE OR REPLACE TEMP TABLE retra_ids AS SELECT DISTINCT CAST(id AS VARCHAR) AS id FROM read_csv_auto('{retra_path}')")
n_retra = con.execute("SELECT COUNT(*) FROM retra_ids").fetchone()[0]
print(f"撤稿论文总数: {n_retra:,}")

撤稿论文总数: 42,188


## Step 2/4: 匹配

扫描 52G PaperReferences.csv, 只保留 reference_ids 在撤稿名单中的行
PaperReferences.csv 两列: id (论文) 引用 reference_ids (被引文献)

In [15]:
refs_path = '/data6/Data1/DATA/Dimensions2024/20240101/PaperReferences.csv'

print("正在扫描 PaperReferences.csv (52G)... 预计 5-10 分钟")
t0 = time.time()

con.execute(f"CREATE OR REPLACE TABLE df_containretra AS SELECT pr.id, pr.reference_ids FROM read_csv_auto('{refs_path}') AS pr WHERE pr.reference_ids IN (SELECT id FROM retra_ids)")

elapsed = time.time() - t0
print(f"完成! 耗时: {elapsed/60:.1f} 分钟")

count = con.execute("SELECT COUNT(*) FROM df_containretra").fetchone()[0]
unique_i = con.execute("SELECT COUNT(DISTINCT id) FROM df_containretra").fetchone()[0]
print(f"匹配行数: {count:,}")
print(f"唯一 focal paper (i): {unique_i:,}")
print(f"平均每个 i 引用撤稿文献数: {count/unique_i:.2f}")

正在扫描 PaperReferences.csv (52G)... 预计 5-10 分钟


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

完成! 耗时: 0.6 分钟
匹配行数: 765,839
唯一 focal paper (i): 647,439
平均每个 i 引用撤稿文献数: 1.18


## Step 3/4: 保存

In [16]:
os.makedirs('/Data4/yutao_wen/processed', exist_ok=True)
output_path = '/Data4/yutao_wen/processed/df_containretra.csv'
con.execute(f"COPY df_containretra TO '{output_path}' (HEADER, DELIMITER ',')")
size_mb = os.path.getsize(output_path) / 1024 / 1024
print(f"已保存: {output_path} ({size_mb:.1f} MB)")

已保存: /Data4/yutao_wen/processed/df_containretra.csv (21.9 MB)


## Step 4/4: 删除汇报

PaperReferences 全量约 20 亿行。保留条件: reference_ids 属于撤稿名单。
删除: reference_ids 不在撤稿名单的普通引用关系。

## 验证: 预览前 5 行

In [17]:
sample = con.execute("SELECT * FROM df_containretra LIMIT 5").df()
print(sample)
# con.close() — 已移除, Step 2 还需要此连接
print("Step 1 完成!")

               id   reference_ids
0  pub.1072932118  pub.1006216542
1  pub.1149626211  pub.1127581668
2  pub.1143588940  pub.1002621477
3  pub.1007418494  pub.1024016961
4  pub.1015571816  pub.1062460108
Step 1 完成!


In [18]:
# 确保 DuckDB 连接还在 (如果上一步误关了内核或连接, 这里重新打开)
try:
    con.execute('SELECT 1')
    print('DuckDB 连接正常')
except:
    import duckdb
    con = duckdb.connect()
    con.execute("SET memory_limit = '100GB'")
    print('DuckDB 已重新连接')


DuckDB 连接正常


---

# Step 2: 补上撤稿论文和被引论文的元信息

**目标:** 给 df_containretra 加上:
- 撤稿论文 r: RPYear, RYear, RJournal_id
- focal paper i: Year, Journal_id


## 2a: 匹配撤稿论文信息

从 matched_output2.csv (50M) 按 reference_ids merge

In [24]:
print("加载撤稿论文信息...")
df_retra = pd.read_csv('/Data4/yutao_wen/matched_output2.csv',
    usecols=['id', 'Year', 'RYear', 'journal.id'])
df_retra = df_retra.rename(columns={
    'id': 'reference_ids', 'Year': 'RPYear', 'journal.id': 'RJournal_id'})
print(f'撤稿论文信息: {len(df_retra):,} 行')

con.execute("CREATE OR REPLACE TEMP TABLE retra_info AS SELECT * FROM df_retra")

con.execute("DROP TABLE IF EXISTS df_containretra2")
con.execute("""
    CREATE OR REPLACE TABLE df_containretra2 AS
    SELECT c.id, c.reference_ids,
           r.RPYear, r.RYear, r.RJournal_id
    FROM df_containretra c
    LEFT JOIN retra_info r ON c.reference_ids = r.reference_ids
""")

total = con.execute("SELECT COUNT(*) FROM df_containretra2").fetchone()[0]
matched = con.execute("SELECT COUNT(*) FROM df_containretra2 WHERE RPYear IS NOT NULL").fetchone()[0]
print(f'总行数: {total:,}')
print(f'匹配到撤稿信息: {matched:,} ({matched/total*100:.1f}%)')
print(f'未匹配: {total-matched:,} ({(total-matched)/total*100:.1f}%)')


加载撤稿论文信息...
撤稿论文信息: 43,435 行
总行数: 766,253
匹配到撤稿信息: 762,268 (99.5%)
未匹配: 3,985 (0.5%)


## 2b: 匹配 focal paper 自己的信息

从 Papers.csv (33G) 取 date_normal → Year, journal.id → Journal_id

In [26]:
# 使用 pandas 读取 Papers.csv，只取需要的 3 列（与旧版 notebook 相同方式）
# pandas 的 errors='coerce' 能安全处理日期缺失值
papers_path = '/data6/Data1/DATA/Dimensions2024/20240101/Papers.csv'

print('正在从 Papers.csv (33G) 提取 Year 和 Journal_id...')
print('  预计 5-10 分钟，使用 pandas read_csv + usecols')
t0 = time.time()

# 与旧版 notebook 完全一致的读取方式
df_papers = pd.read_csv(papers_path,
    usecols=['id', 'date_normal', 'journal.id'])

print(f'  读取完成，耗时: {(time.time()-t0)/60:.1f} 分钟')
print(f'  共 {len(df_papers):,} 行')

# 提取年份（与旧版完全一致：errors='coerce' 安全处理无效日期）
df_papers['Year'] = pd.to_datetime(
    df_papers['date_normal'], errors='coerce').dt.year

# 丢掉无法解析年份的行
before = len(df_papers)
df_papers = df_papers.dropna(subset=['Year', 'journal.id'])
df_papers['Year'] = df_papers['Year'].astype(int)
after = len(df_papers)
print(f'  删除缺失 Year/journal.id 的行: {before - after:,} → 保留 {after:,}')

# 注册到 DuckDB 并合并
df_papers = df_papers.rename(columns={'journal.id': 'Journal_id'})
con.execute("CREATE OR REPLACE TEMP TABLE focal_info_clean AS SELECT * FROM df_papers")

con.execute("DROP TABLE IF EXISTS df_containretra2_enriched")
con.execute("""
    CREATE TABLE df_containretra2_enriched AS
    SELECT c.*, f.Year, f.Journal_id
    FROM df_containretra2 c
    LEFT JOIN focal_info_clean f ON c.id = f.id
""")

total2 = con.execute("SELECT COUNT(*) FROM df_containretra2_enriched").fetchone()[0]
has_year = con.execute("SELECT COUNT(*) FROM df_containretra2_enriched WHERE Year IS NOT NULL").fetchone()[0]
print(f'\n总行数: {total2:,}')
print(f'有 Year 信息: {has_year:,} ({has_year/total2*100:.1f}%)')
print(f'缺失 Year: {total2-has_year:,}')


正在从 Papers.csv (33G) 提取 Year 和 Journal_id...
  预计 5-10 分钟，使用 pandas read_csv + usecols
  读取完成，耗时: 7.8 分钟
  共 141,623,211 行
  删除缺失 Year/journal.id 的行: 24,928,059 → 保留 116,695,152


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


总行数: 766,253
有 Year 信息: 694,579 (90.6%)
缺失 Year: 71,674


## 2c: 计算 RYear - Year

如果 RYear < Year, 论文在撤稿后才引用 → 异常数据, 待删

In [27]:
con.execute("DROP TABLE IF EXISTS df_containretra2_final")
con.execute("""
    CREATE OR REPLACE TABLE df_containretra2_final AS
    SELECT *, (RYear - Year) AS "RYear-Year"
    FROM df_containretra2_enriched
""")

result = con.execute("""
    SELECT COUNT(*) AS total,
        SUM(CASE WHEN "RYear-Year" >= 0 THEN 1 ELSE 0 END) AS valid,
        SUM(CASE WHEN "RYear-Year" < 0 THEN 1 ELSE 0 END) AS neg,
        SUM(CASE WHEN "RYear-Year" IS NULL THEN 1 ELSE 0 END) AS null_val
    FROM df_containretra2_final
""").fetchone()

print(f'总行数: {result[0]:,}')
print(f'RYear-Year >= 0 (正常, 引用在撤稿前): {result[1]:,}')
print(f'RYear-Year < 0 (异常, 引用时已撤稿): {result[2]:,}')
print(f'RYear-Year NULL (缺失): {result[3]:,}')

print("预览前5行:")
print(con.execute("SELECT * FROM df_containretra2_final LIMIT 5").df())


总行数: 766,253
RYear-Year >= 0 (正常, 引用在撤稿前): 510,321
RYear-Year < 0 (异常, 引用时已撤稿): 179,877
RYear-Year NULL (缺失): 76,055
预览前5行:
               id   reference_ids  RPYear   RYear   RJournal_id  Year  \
0  pub.1005848756  pub.1043667119  2014.0  2023.0  jour.1092493  2016   
1  pub.1013407133  pub.1025335416  2012.0  2014.0  jour.1043768  2013   
2  pub.1157126552  pub.1134476360  2021.0  2023.0  jour.1077219  2023   
3  pub.1132777327  pub.1006345487  2014.0  2016.0  jour.1312353  2020   
4  pub.1026699624  pub.1025204398  2006.0  2013.0  jour.1002377  2011   

     Journal_id  RYear-Year  
0  jour.1038393         7.0  
1  jour.1092625         1.0  
2  jour.1045337         0.0  
3  jour.1038765        -4.0  
4  jour.1041633         2.0  


## 2d: 保存 df_containretra2

输出到 /Data4/yutao_wen/processed/

In [28]:
output_path = '/Data4/yutao_wen/processed/df_containretra2.csv'
con.execute(f"COPY df_containretra2_final TO '{output_path}' (HEADER, DELIMITER ',')")
import os
size_mb = os.path.getsize(output_path) / 1024 / 1024
n_rows = con.execute("SELECT COUNT(*) FROM df_containretra2_final").fetchone()[0]
print(f'已保存: {output_path}')
print(f'行数: {n_rows:,}, 大小: {size_mb:.1f} MB')
print('Step 2 完成! 下一步: Step 3 清洗筛选')


已保存: /Data4/yutao_wen/processed/df_containretra2.csv
行数: 766,253, 大小: 56.3 MB
Step 2 完成! 下一步: Step 3 清洗筛选
